**GenAI disclosure:** this practice notebook (not graded) was filled in with help from Claude (Anthropic) as a worked example; outputs in Q9-Q11 come from a local run of Qwen3-1.7B on the first 10 training samples.

## Conversational AI (25/26) Assignment 1: LLM inference in task-oriented dialogues

LLMs are being widely deployed as dialogue systems, e.g. QA systems. 
In this exercise, we will be loading a large language model (Qwen3) and using it for task-oriented conversational modeling with subjective knowledge. 

The DSTC11 Task 5 and its dataset are explained in detail [here](https://github.com/alexa/dstc11-track5). Also, you can read more about it in the [paper](https://arxiv.org/pdf/2305.12091).

In essence, the task is to respond to subjective user requests regarding hotels and restaurants, e.g., "does the restaurant have a nice vibe?". Our conversational model has to generate a response to this question based on conversation history and previous reviews it has access to. 

This task consists of three sub-tasks: 
1. Knowledge-seeking Turn Detection
2. Knowledge Selection
3. **Knowledge-grounded Response Generation** (our focus)

We will be focusing on the last sub-task, _knowledge-grounded response generation_, for which we need the dialog history, request, and knowledge. With these as the input, the model should then generate a response.

**This notebook contains the following parts:**
1. Exploring the DSTC11 Task5 data and formatting it in the proper structure
2. Off-the-shelf inference with Qwen3-1.7B
3. Analyzing the model behavior

## Exercises

We will be using the dialogue history between a user and a system to have Qwen generate a response to a knowledge-seeking request from the user. 
The goal is to load an LLM and understand the strengths and limitations of its off-the-shelf usage for task-oriented dialogues.

Throughout the notebook, you will find questions you need to answer to complete the exercises. These are both coding questions and questions that evaluate your understanding of the data, the process, and the model behavior. These questions are indicated as  **<span style="background:yellow">Q#</span>**.

**Exercise steps:**
1. Load the dataset and understand its structure, individual components, what the inputs should be and what the output should be. **(Q1)**
2. Convert the dataset to a structure that is useful for our LLM to generate its responses with. **(Q2 - Q4)**
3. Answer questions about key concepts in this exercise. **(Q5 - Q8)**
5. Answer questions about your implementation and manual analysis of 10 samples. **(Q9 - Q11)**
    

## A couple of notes about Kaggle notebooks
* [Terms and conditions](https://www.kaggle.com/terms)
* Short note on the [GPU limits](https://www.kaggle.com/discussions/general/108481)
* [Uploading your own model to Kaggle](https://www.kaggle.com/discussions/questions-and-answers/63328)

**Kaggle directories** may be confusing because they are present on another machine that we have no access to (effectively "in the cloud"). 
To help your orientation, keep this in mind:
* Input data files are available in the read-only "/kaggle/input/" directory
* You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
* You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Kaggle seems to run smoothly, including in terms of GPUs. There are two common environment issues that happen because we forget to turn on the internet or the GPUs. Specifically:
* `ERROR: Could not find a version that satisfies the requirement ... (from versions: none) ERROR: No matching distribution found for ...` - to solve this issue, turn your notebook Internet ON.
Make sure your accelerator is set to `GPU P100` or `GPU T4x2`

**<span style="color:red">Important</span>**

These are five good practices when working with Kaggle notebooks:
1. Pay attention to usage statistics, especially memory, CPU, and GPU
2. Pay attention to the quota of GPU (measured in hours)
3. "Turn OFF" the internet after each session. Turn on the internet when starting a session.
4. "Turn off" the accelerator after each use. Turn the accelerator on when starting a session.
5. Save a version after you make changes. This ensures that your teammate can see the latest changes. If you get a question from Kaggle about versions, you can revert to the latest version.

## Installation

The Kaggle notebooks use a Python 3 environment, and they are already "pre-loaded" with various analytic Python packages, like Json, Pandas, and Numpy. If you are curious, you can see the package definition in [this repository](https://github.com/kaggle/docker-python).

We will install the Transformers package for working with language models.

In [ ]:
!pip install transformers==4.53.3

## Imports

The following code loads several standard packages and packages for working with LLMs.

In [ ]:
import numpy as np 
import json
import os
import shutil
import subprocess
import sys
from typing import List
from datasets import Dataset


from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification, pipeline

## Clone the data/task repository

We start by cloning the data and task repository and changing the working directory to that directory.

In [ ]:
def setup_repo(repo_url: str, repo_name: str, work_dir: str = "/kaggle/working"):
    os.chdir(work_dir)
    
    # Remove repo if it exists
    if os.path.exists(os.path.join(work_dir, repo_name)):
        shutil.rmtree(os.path.join(work_dir, repo_name))
    
    # Clone repo
    subprocess.run(["git", "clone", repo_url], check=True)
    
    # Move into repo/data
    os.chdir(os.path.join(repo_name, "data"))


setup_repo("https://github.com/lkra/dstc11-track5.git", "dstc11-track5")

## Exploring the data
The data we will use contains these essential components: 
- __knowledge__: The reviews for the corresponding restaurant or hotel (the data also contains FAQs - we won't be using those!)
- __dialogue__: The dialogue history between the user and the system
- __response__: The ground-truth response we expect from the system

Let's list all files in the current directory iteratively and then load the data first to understand it better!

In [ ]:
## List all files in the current directory iteratively:
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Loading the data: For this exercise, we will only use the __training data__. So let's load it.

In [ ]:
with open('train/logs.json', 'r') as f:
    train_ds=json.load(f)

Let's look at the first example:

In [ ]:
sample = train_ds[0]
sample

## Conversation History
The first index is the dialogue, clearly marking what the user and system have said until now. The last dictionary in this list is the last utterance from the dialogue, indicating the query that the model should respond to. 

<span style="background:yellow">__Q1:__ In the block below, write the code to first access the dialogue from _sample_, and then the query:</span>

In [ ]:
dialogue = sample                 # the whole sample is the list of turns
query = sample[-1]["text"]        # the last user turn is the knowledge-seeking request

dialogue, query

<span style="background:yellow">__Q2:__ To make it more readable for the model (and also ourselves), let's reformat the conversation history to the following:</span>

>[
>{'role': 'user', 'content': 'Can you help me find a place to stay that is moderately priced and includes free wifi?'},
>{'role': 'system', 'content': 'sure, i have 17 options for you'}
>...]

Formats a list of dialogue by turning it into a readable list.

Given a list of dictionaries, each representing a dialogue turn, this function formats the dialogue by prefixing
each turn with either "user" or "system" as the role based on the speaker. The speaker is identified by the 'speaker' key in the dictionary: 'U' for user, else it is the system.

In [ ]:
def format_dialogue(sample: List[dict]) -> List[dict]:
    """
    Args:
    sample (List[dict]): A list of dictionaries where each dictionary contains two keys:
        - 'speaker' (str): A string indicating the speaker of the turn ('U' for user, 'S' for system).
        - 'text' (str): The text spoken by the respective speaker.

    Returns:
        List[dict]: A new array with a specific role and content

    """
    messages=[]
    messages.append({"role": "system", "content": "You are an assistant."})
    for dialogue_element in sample:
        role = "user" if dialogue_element["speaker"] == "U" else "system"
        messages.append({"role": role, "content": dialogue_element["text"]})

    return messages


print(format_dialogue(dialogue))

### Knowledge

Let's look at the knowledge, which contains the reviews for the conversation:

In [ ]:
## first load the labels
with open('train/labels.json', 'r') as f:
    labels=json.load(f)

From the labels, load the necessary knowledge, and print it. 

In [ ]:
knowledge_sample_list = labels[0]["knowledge"]
print(knowledge_sample_list)

As you can see, here we just have some entries provided. We need to extract the knowledge from the knowledge base using these entries.

So, let's now load the knowledge base.

In [ ]:
with open('knowledge.json', 'r') as f:
    knowledge_base=json.load(f)

This is again a list of different reviews. If we were to give this knowledge directly to the LLM, we are giving it extra information that is not necessary for generating the response. We only need the review _itself_. 

<span style="background:yellow">__Q3:__ In the block below, write a function that takes as input the knowledge of one sample and outputs the reviews in a list:</span>

This function extracts and returns a list of review sentences from the given knowledge data.

Given a list of dictionaries representing knowledge data, this function collects review text from each dictionary.

In [ ]:
def get_reviews(knowledge: List[dict]) -> List[str]:
    """
    Args:
        knowledge (List[dict]): A list of dictionaries containing review information.

    Returns:
        List[str]: A list of strings where each string is a review extracted from the knowledge data.

    """
    sources = []
    for k in knowledge:
        try:
            domain = k["domain"]                 # "hotel" or "restaurant"
            entity_id = k["entity_id"]           # which hotel/restaurant
            doc_id = k["doc_id"]                 # which review of that entity
            sent_id = k["sent_id"]               # which sentence of that review (FAQ entries have none -> skipped)

            source = knowledge_base[domain][str(entity_id)]["reviews"][str(doc_id)]["sentences"][str(sent_id)]
            sources.append(source)
        except:
            continue

    return sources


print(get_reviews(knowledge_sample_list))

### Response (Ground-truth)
Similarly, we can load the ground-truth response. 

In [ ]:
response = labels[0]['response']
response

## Create Dataset
Until now we have looked at and applied our functions to only one sample.

<span style="background:yellow">__Q4:__ In the block below, write a function that takes the content and labels as input and creates a dataset.</span>

Let us now correctly format our entire dataset! Essentially, we want our nicely formatted dialogue, knowledge, and the ground-truth response. We can do this as follows: 

__Important__: In this step, you should use the previously defined functions and the loaded data. Most of your work here involves calling those functions or using the provided data. Avoid starting from scratch.

In [ ]:
def reformat_dataset(dataset, labels_dataset):
    reformatted_dataset = {
        "dialogue": [],
        "knowledge": [],
        "response": [],
    }
    for sample_index in range(len(dataset)):
        try:
            sample_dialogue = format_dialogue(dataset[sample_index])
            sample_knowledge = get_reviews(labels_dataset[sample_index]["knowledge"])   # non-target turns have no "knowledge" -> skipped
            sample_response = labels_dataset[sample_index]["response"]

            reformatted_dataset["dialogue"].append(sample_dialogue)
            reformatted_dataset["knowledge"].append(sample_knowledge)
            reformatted_dataset["response"].append(sample_response)
        except:
            continue

    return reformatted_dataset

reformatted_dataset = reformat_dataset(train_ds, labels)
dataset = Dataset.from_dict(reformatted_dataset)
dataset

Now if we access the first sample, it is much more readable and we can access the different components directly!

In [ ]:
dataset[0]

## Off-the-shelf usage

To see what our LLM does without any adaptation, let's run inference directly on a couple of samples!
We first need to load our model. 

In [ ]:
model_name = "Qwen/Qwen3-1.7B"
qwen3_tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen3_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

What exactly happened in the previous cell?

We did two things.

First, we loaded a tokenizer for the pretrained model Qwen3-1.7B. It is called `AutoTokenizer` since the tokenizer is chosen automatically from the model architecture.

Then, `AutoModelForCausalLM` loads the pretrained Qwen model with a so-called "causal language modeling head" for text generation. By adding a task head to predict the next token, this class suits tasks like creative writing, conversational chat, and code completion. The approach is "autoregressive" because the generation is done one token at a time rather than all at once.
Like AutoTokenizer, AutoModelForCausalLM automatically selects the model architecture from the model class.

Now, let’s generate the responses and compare them to the ground-truth answers.

In [ ]:
messages = dataset[0]['dialogue']

text = qwen3_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
model_inputs = qwen3_tokenizer([text], return_tensors="pt").to(qwen3_model.device)

generated_ids = qwen3_model.generate(**model_inputs, max_new_tokens=500)
output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]

print("MODEL: ", qwen3_tokenizer.decode(output_ids, skip_special_tokens=True).strip())
print("Ground-truth: ", dataset[0]["response"])

So this cell applied the two class instantiations: `qwen3_tokenizer (AutoTokenizer)` and `qwen3_model (AutoModelForCausalLM)`. 

`qwen3_tokenizer` is used to convert the text into numerical tensors (token ids), stored into the variable `model_inputs`. 

Then, we use the `generate()` function of our `qwen3_model` with the tokenized input (`model_inputs`) as an argument.
Note that the **generation is based on token tensors and not the raw text**!

Afterward, the `generated_ids` are transformed into `output_ids` and then into text using the `decode()` function.

We can now test the same for other samples, for example, the ones with id 1 and 20.

In [ ]:
messages = dataset[1]['dialogue']

text = qwen3_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
model_inputs = qwen3_tokenizer([text], return_tensors="pt").to(qwen3_model.device)

generated_ids = qwen3_model.generate(**model_inputs, max_new_tokens=500)
output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]

print("MODEL: ", qwen3_tokenizer.decode(output_ids, skip_special_tokens=True).strip())
print("Ground-truth: ", dataset[1]["response"])

In [ ]:
messages = dataset[20]['dialogue']

text = qwen3_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
model_inputs = qwen3_tokenizer([text], return_tensors="pt").to(qwen3_model.device)

generated_ids = qwen3_model.generate(**model_inputs, max_new_tokens=500)
output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]

print("MODEL: ", qwen3_tokenizer.decode(output_ids, skip_special_tokens=True).strip())
print("Ground-truth: ", dataset[20]["response"])

Hooray! We were able to prompt our Qwen model with multiple inputs, transform them into tokens, and generate outputs in a legible way. Then, we can compare these outputs to those in the ground truth to understand how close the model generation is to the ground truth. Don't worry about doing this one dialogue at a time for now, and about comparing manually. We will dive into those topics later in the course.

Before we finish the coding explanation, a few notes about specific parameters above:
* `add_generation_prompt=True` adds an assistant template header (e.g., <|im_start|>assistant\n) to the end of the formatted string, telling the model where to begin generating its answer.
* `enable_thinking=False` disables the model's internal thinking/reasoning template tags (e.g., <think>...</think>), directing the model to output a direct response rather than generating hidden chain-of-thought tokens. We'll cover reasoning models later in the course.
* `max_new_tokens=500` sets the limit of newly generated tokens to 500. The prompt length is not counted in this limit! Note that the generation stops earlier if the model outputs an EOS (end of sequence) token.
* `skip_special_tokens=True` filters out special control tokens (such as <|im_start|>, <|im_end|>, or EOS tokens). Because of this flag, the `decode()` function returns only plain conversational text.

We hope that this notebook was insightful and fun. 

If anything in it is confusing or unclear, remember you can use office hours to ask questions about the notebooks!

## Understanding questions

<span style="background:yellow">__Q5:__ Describe what the classes `AutoTokenizer` and `AutoModelForCausalLM` are used for. </span> 

**Answer:** `AutoTokenizer` turns text into the token ids the model works with, and back again. `from_pretrained("Qwen/Qwen3-1.7B")` picks the tokenizer that belongs to that checkpoint, including its vocabulary and its chat template. For example, `apply_chat_template` turns our list of `{role, content}` dicts into one string with `<|im_start|>user ... <|im_end|>` markers, and calling the tokenizer on that string gives a tensor of ids. `AutoModelForCausalLM` loads the pretrained weights with a causal language-modelling head: a layer that, for every position, gives a probability for each vocabulary token being the *next* token. "Auto" means the right architecture class (here Qwen3) is chosen from the checkpoint's config, so the same code works for other decoder-only LLMs.

<span style="background:yellow">__Q6:__ Explain the line `generated_ids = qwen3_model.generate(**model_inputs, max_new_tokens=500)`. What are the function arguments, and what is the expected output?</span> 

**Answer:** `model_inputs` is the tokenizer output, a dict with `input_ids` (the prompt as token ids) and `attention_mask` (1 for real tokens). `**` unpacks it into keyword arguments. `generate()` then decodes autoregressively: it predicts one token, appends it to the input, and repeats until it emits the end-of-sequence token (`<|im_end|>`) or reaches `max_new_tokens=500` new tokens; the prompt does not count towards the 500. Which token is picked comes from the model's default generation config (for Qwen3: sampling with temperature/top-p), so two runs can give different answers. The output is a tensor of shape `[batch, prompt_len + new_len]` holding the prompt ids *followed by* the generated ids. That is why the next line slices off `model_inputs.input_ids.shape[1]` tokens before calling `decode()`.

Please answer each question with a single paragraph, consisting of 2-5 sentences. Include examples to support your answers.

<span style="background:yellow">__Q7:__ Describe the dataset in detail, focusing on its structure, domains, and size. What is the meaning of `query` and `knowledge` in this context?</span> 

**Answer:** The data is the DSTC11 Track 5 training split. It is built on MultiWOZ-style task-oriented dialogues in two domains, **hotels** and **restaurants**, in Cambridge. `logs.json` has **32,604** dialogue contexts, each a list of turns `{speaker: 'U'|'S', text}` (3 to 45 turns, mean 8.3), and `labels.json` has one label per context. Only **16,897** contexts have `target: true`, meaning the last user turn asks for subjective knowledge (9,295 hotel, 7,602 restaurant). These are the rows that end up in our dataset; the other 15,707 are plain booking turns without knowledge or a response. Each target label points to review sentences by `domain`, `entity_id`, `doc_id` and `sent_id`, and `knowledge.json` stores the reviews (and FAQs) of 33 hotels and 110 restaurants. A sample uses about 3.6 review sentences on average (84 rows point only to FAQs, so they have no review sentences). The **query** is the last user turn, the request the system must answer, e.g. *"I have back issues. Does this place have comfortable beds?"*. The **knowledge** is the set of review sentences that are relevant to that query, e.g. *"the beds were really uncomfortable"* from guests of the Bridge Guest House. A good answer summarises them: *"...known for having pretty uncomfortable beds according to most guests. Only one guest found it to be comfortable."*

<span style="background:yellow">__Q8:__ What data pre-processing steps did we take and why?</span> 

**Answer:** (1) We kept only the turns we need: the last user turn is the query and everything before it is the history. (2) We converted the dialogue from `{speaker: 'U'/'S', text}` into chat messages `{role, content}` and added a system prompt ("You are an assistant."), because `apply_chat_template` expects this format and Qwen was instruction-tuned on it. (3) We resolved the knowledge pointers (`domain`, `entity_id`, `doc_id`, `sent_id`) into the actual review sentences. We dropped the metadata (traveler type, ids) and FAQ entries, so the model would only get the text that matters, without noise or wasted context. (4) We joined dialogue, reviews and the gold response per sample and dropped non-target turns, which have no knowledge or response. This turned three loosely linked JSON files into one Hugging Face `Dataset` with clean, aligned columns, so each sample can be fed to the model and compared with the ground truth in one line.

## Analyze Model Responses
Let's look at what types of generations the LLM produces. For this, pick the first **10** samples for the analysis. 
Then, based on your observations, answer the following 5 questions (remember to give examples to illustrate your findings).


#### Questions

<span style="background:yellow">__Q9:__ What do you observe about the quality of the generated responses? Do they effectively address the query? 
Explain what data components were considered by the LLM to generate the response. What role do reviews, as a source of knowledge, play in generating the responses?</span> 

**Answer:** The answers are fluent and polite, but most do not really answer the question with the right content. The generation cells only pass `dataset[i]['dialogue']`, so the model sees the system prompt, the history and the query, **but never the reviews**. It answers from its prior beliefs and from what the history implies. That leads to three patterns in the first 10 samples. (1) **Confident hallucination**: #0 "Yes, the Bridge Guesthouse has comfortable beds... soft, supportive mattresses", while 4 of the 5 review sentences say the bed is uncomfortable; #6 says the Bedouin is "in the city centre" while the only review says the location is poor. (2) **Refusal plus invented alternatives**: #2 says it has no information and then recommends made-up hotels ("Hotel Lido", "Hotel Plaza"); #9 suggests "The Ritz-Carlton". (3) **Lucky hits**: #1 (outdoor dining) and #8 (friendly staff) match the gold answer, but only because the reviews happen to be positive and the model is biased to say yes. The reviews are supposed to be the evidence the answer is grounded in: the gold answers summarise them ("most guests... only one found it comfortable"). Without them, the model cannot know the subjective facts, so its correctness is basically chance. To fix this, the reviews must be put into the prompt (e.g. as a system message or before the query), which is retrieval-augmented generation.

<span style="background:yellow">__Q10:__ How does the LLM perform in the following scenario?</span> 
1. When all reviews agree with each other
2. When only one review disagrees
3. When opinions in the reviews are mixed (i.e., high disagreement)

**Answer:** Because the model never sees the reviews, the level of agreement does not change what it does. What changes is whether its default ("yes, it's great") happens to be right. (1) **All reviews agree (positive)**: #1, #8 and partly #20. The answers match the gold in #1 and #8 ("Yes, the staff at the Lovell Lodge are friendly"), but #3 and #20 refuse ("I don't have the capability to check the staff's friendliness") even though all reviews are positive, so agreement alone does not help. (2) **One review disagrees**: #0, where 3 of 4 bed sentences are negative and one is positive. The model answers the *minority* view with full confidence ("comfortable beds"), the opposite of the gold "pretty uncomfortable according to most guests". (3) **Mixed opinions**: #2, #5 and #7. The gold answers say "some say... while others say". The model never produces this balanced pattern: it refuses (#2), drifts off topic (#5 talks about restaurants in the city of Bangkok instead of the restaurant *Bangkok City*), or praises a different hotel (#7 "the Finches... many guests have praised its friendly staff"). Side note: in #4 the gold answer itself ("incredible view") does not match the mostly negative review sentences, so the ground truth is noisy too.

<span style="background:yellow">__Q11:__ Compare the behavior of the model for dialogues of different length. Do you observe any impact of the length of the dialogue/conversation history on the LLM's generation?</span> 


**Answer:** With only 10 samples (3 to 13 turns, 69 to 374 prompt tokens) this is a weak trend, but length does matter. **Very short histories** give too little context: #5 has 3 turns and only the restaurant name "Bangkok City" to go on, and the model reads it as the city Bangkok and gives generic travel advice. **Medium histories (5 to 7 turns)** produce the most confident, on-topic answers (#0, #4, #6, #8), right or wrong, because the entity is clearly named in the last system turn. **Long histories (11 to 13 turns: #3, #7, #20)** lead to more refusals and entity confusion: in #7 the model talks about "the Finches", a hotel mentioned earlier in the conversation, instead of the Cambridge Belfry being discussed, and in #3 it switches to offering a booking (an earlier task in the dialogue) instead of answering. So a longer history adds distracting entities and earlier sub-tasks that a small 1.7B model mixes up. One more factor: we pass the earlier system turns with role `system`, so the model reads them as instructions rather than as its own earlier replies (`assistant`), which may add to the confusion in long dialogues.

## Example quiz question from this notebook

**E1: What is the consequence of setting enable_thinking=False in the context of newer 'reasoning' models like Qwen-Thinking or DeepSeek-R1?**

(A) Reduces the floating-point precision

(B) Disables the attention mechanism

(C) Suppresses internal chain-of-thought

(D) Stops the model from learning

**Answer: (C)**

**E2: What does setting add_generation_prompt=True in the apply_chat_template method achieve?**

(A) Enables multi-turn history

(B) Includes system instructions

(C) Automatically tokenizes text

(D) Appends an assistant header

**Answer: (D)**
